In [0]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import uuid

spark = SparkSession.builder.getOrCreate()

def ingest_to_bronze(source_path: str, table_name: str, batch_id: str):
    """
    Realiza a ingestão bruta para a camada Bronze
    """
    print(f"Iniciando ingestão da tabela: bronze_{table_name}")
    
    # Leitura dos dados brutos preservando o schema original
    df_raw = spark.read.option("header", True).option("inferSchema", True).csv(source_path)
    colunas_originais = df_raw.columns
    
    # Adiciona metadados de auditoria e rastreabilidade
    df_bronze = (
        df_raw
        .withColumn("arquivo_origem", F.col("_metadata.file_path"))
        .withColumn("data_ingestao", F.current_date())
        .withColumn("timestamp_ingestao", F.current_timestamp())
        .withColumn("batch_id", F.lit(batch_id))
        .withColumn("hash_linha", F.md5(F.concat_ws("||", *[F.col(c).cast("string") for c in colunas_originais])))
        .withColumn("schema_version", F.lit("1.0"))
    )
    
    # Persistência incremental em formato Delta com evolução de schema habilitada
    (df_bronze.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(f"workspace.default.bronze_{table_name}")
    )
    
    print(f"Sucesso! Tabela workspace.default.bronze_{table_name} atualizada.")

# Execução do pipeline de ingestão
meu_batch_id = str(uuid.uuid4())
volume_path = "/Volumes/workspace/default/raw_data"

ingest_to_bronze(f"{volume_path}/clientes_cdc.csv", "clientes", meu_batch_id)
ingest_to_bronze(f"{volume_path}/contas_cdc.csv", "contas", meu_batch_id)
ingest_to_bronze(f"{volume_path}/cartoes_cdc.csv", "cartoes", meu_batch_id)